# Node2Vec Embedding Generation

This notebook generates structural node embeddings using the Node2Vec algorithm on the WikiCS graph. Unlike text-based embeddings, these capture the connectivity patterns of the nodes.

**Output:** `master_embeddings.parquet` with columns `id` (original node id) and `embedding` (dense vector).

In [1]:
import json
import os
import torch
import pandas as pd
import numpy as np
from torch_geometric.nn import Node2Vec
from tqdm import tqdm

c:\Users\GSU\Desktop\python projects\21401920-final-project\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# Load the dataset
DATA_PATH = "../../data/data-embeddings.json"
with open(DATA_PATH, "r") as f:
    data = json.load(f)

nodes = data["nodes"]
print(f"Total nodes: {len(nodes)}")

Total nodes: 11701


In [3]:
# Create ID mapping and build edge list
id_to_idx = {node["id"]: i for i, node in enumerate(nodes)}
idx_to_id = {i: node["id"] for i, node in enumerate(nodes)}

edge_list = []
for node in nodes:
    u_idx = id_to_idx[node["id"]]
    for v_id in node.get("outlinks", []):
        if v_id in id_to_idx:
            v_idx = id_to_idx[v_id]
            edge_list.append([u_idx, v_idx])

edge_index = torch.tensor(edge_list, dtype=torch.long).t().contiguous()
print(f"Total edges: {edge_index.size(1)}")

Total edges: 297110


In [4]:
# Initialize Node2Vec model
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Using device: {device}")

model = Node2Vec(
    edge_index,
    embedding_dim=128,
    walk_length=20,
    context_size=10,
    walks_per_node=10,
    num_negative_samples=1,
    p=1.0,
    q=1.0,
    sparse=True
).to(device)

loader = model.loader(batch_size=128, shuffle=True, num_workers=0)

Using device: cuda


In [5]:
# Training loop
optimizer = torch.optim.SparseAdam(list(model.parameters()), lr=0.01)

def train():
    model.train()
    total_loss = 0
    for pos_rw, neg_rw in loader:
        optimizer.zero_grad()
        loss = model.loss(pos_rw.to(device), neg_rw.to(device))
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    return total_loss / len(loader)

epochs = 50
for epoch in range(1, epochs + 1):
    loss = train()
    if epoch % 10 == 0 or epoch == 1:
        print(f'Epoch: {epoch:03d}, Loss: {loss:.4f}')

Epoch: 001, Loss: 6.9752
Epoch: 010, Loss: 0.9525
Epoch: 020, Loss: 0.9235
Epoch: 030, Loss: 0.9207
Epoch: 040, Loss: 0.9191
Epoch: 050, Loss: 0.9189


In [6]:
# Extract and save embeddings
@torch.no_grad()
def save_embeddings():
    model.eval()
    z = model.to('cpu')()
    
    output_data = []
    for i in range(z.size(0)):
        output_data.append({
            "id": idx_to_id[i],
            "embedding": z[i].numpy()
        })
    
    df = pd.DataFrame(output_data)
    df.to_parquet("master_embeddings.parquet", index=False)
    print(f"Saved {len(df)} embeddings to master_embeddings.parquet")

save_embeddings()

Saved 11701 embeddings to master_embeddings.parquet
